# One-Hot Encoding & ColumnTransformer in scikit-learn

A hands-on walkthrough of encoding categorical features for machine learning, covering one-hot encoding with both pandas and scikit-learn, the k-1 dummy variable trick, handling high-cardinality categories, and combining everything into a single reusable pipeline step with `ColumnTransformer`.

**Dataset:** `cars.csv` (used car listings with columns like `brand`, `km_driven`, `fuel`, `owner`, and `selling_price`). Place the CSV in the same folder as this notebook before running it.

## Setup

Import the libraries we'll need and load the dataset.

In [ ]:
import numpy as np
import pandas as pd

In [ ]:
df = pd.read_csv('cars.csv')

## A quick look at the data

Before encoding anything, it's worth checking how many unique categories each categorical column has — that drives which encoding strategy makes sense.

In [ ]:
df.head()

In [ ]:
df['brand'].nunique()

In [ ]:
df['fuel'].value_counts()

In [ ]:
df['owner'].value_counts()

## 1. One-hot encoding with pandas

`pd.get_dummies()` is the fastest way to one-hot encode columns directly on a DataFrame. It creates one binary column per category, which is simple but can blow up the number of columns for high-cardinality features.

In [ ]:
pd.get_dummies(df, columns=['fuel', 'owner'])

## 2. k-1 one-hot encoding (avoiding the dummy variable trap)

If a categorical column has *k* categories, you only need *k-1* dummy columns to represent it — the dropped category is implied when all the others are 0. This avoids multicollinearity between the dummy columns, which matters for linear models. `drop_first=True` does this automatically.

In [ ]:
pd.get_dummies(df, columns=['fuel', 'owner'], drop_first=True)

## One-hot encoding with scikit-learn

Pandas' `get_dummies` is handy for quick exploration, but it doesn't fit into a scikit-learn pipeline and has no `transform`/`fit` separation — meaning it doesn't reliably reproduce the same columns on new data. `OneHotEncoder` solves this: it learns the categories from the training set with `fit`, then applies that exact same encoding to the test set with `transform`, so train and test stay consistent.

In [ ]:
df.head()

### Train/test split

We split first, before fitting any encoder, so the encoder only ever learns categories from the training data — fitting on the full dataset (including test data) would leak information from the test set into training.

In [ ]:
from sklearn.model_selection import train_test_split

x_train, x_test, y_train, y_test = train_test_split(
    df.iloc[:, 0:4], df.iloc[:, -1], test_size=0.2, random_state=True
)

In [ ]:
x_train

### Fit the encoder on the training set only

In [ ]:
from sklearn.preprocessing import OneHotEncoder

In [ ]:
ohe = OneHotEncoder(drop='first', sparse_output=False, dtype=np.int32)

In [ ]:
x_train_new = ohe.fit_transform(x_train[['fuel', 'owner']])

In [ ]:
# transform (not fit_transform!) on the test set, so it reuses the categories
# learned from training data
x_test_new = ohe.transform(x_test[['fuel', 'owner']])

### Stitching the encoded and untouched columns back together

`OneHotEncoder` only touched `fuel` and `owner`. The other columns (`brand`, `km_driven`) still need to be combined with the new encoded columns into one array.

In [ ]:
x_train[['brand', 'km_driven']].values

In [ ]:
x_train_new

In [ ]:
np.hstack((x_train[['brand', 'km_driven']].values, x_train_new)).shape

## 3. One-hot encoding with top categories

`brand` has a lot of unique values, and one-hot encoding all of them would create a huge, mostly-empty (sparse) feature space. A common fix is to keep only the most frequent categories and lump everything else into a single `'uncommon'` bucket.

In [ ]:
counts = df['brand'].value_counts()
threshold = 100

In [ ]:
rep1 = counts[counts < threshold].index

In [ ]:
pd.get_dummies(df['brand'].replace(rep1, 'uncommon'))

## Putting it together with `ColumnTransformer`

Doing each encoding step by hand and manually `hstack`-ing the results works, but it's error-prone and doesn't fit cleanly into a scikit-learn `Pipeline`. `ColumnTransformer` lets us declare *which* transformer applies to *which* columns in one object, and it automatically handles the rest (`remainder='passthrough'` keeps the untouched columns).

In [ ]:
from sklearn.compose import ColumnTransformer

In [ ]:
transformer = ColumnTransformer(transformers=[
    ('tnf1', OneHotEncoder(sparse_output=False, drop='first'), ['brand', 'fuel', 'owner'])
], remainder='passthrough')

Fit the transformer on the training data and apply the same fitted transformer to the test data — same train/test discipline as before, just in one step instead of one per column group.

*(Fixed from the original draft: `fit_transform` was being assigned `.shape` instead of the transformed array, which silently threw away the actual transformed data.)*

In [ ]:
x_train = transformer.fit_transform(x_train)
x_train.shape

In [ ]:
x_test = transformer.transform(x_test)

In [ ]:
x_test = pd.DataFrame(x_test)

Save the processed test set so it can be reused without re-running the pipeline.

In [ ]:
x_test.to_csv('test_dataset.csv', index=False)

In [ ]:
x_test.describe()

## Takeaways

- `pd.get_dummies` is quick for exploration but doesn't separate `fit`/`transform`, so it can't guarantee train and test sets get encoded the same way.
- `OneHotEncoder` fixes that: fit on train, transform on train and test.
- Use `drop='first'` (k-1 encoding) to avoid multicollinearity for linear models.
- For high-cardinality columns, group rare categories together before encoding.
- `ColumnTransformer` bundles multiple column-specific transformations into a single, pipeline-friendly object instead of manual `hstack`ing.